# Lab 6 - Classification

- Họ và tên: Trịnh Hửu Thọ

- MSSV: 22110238

## I. Hướng dẫn

### Khởi tạo Spark

In [1]:
import findspark
findspark.init()

import pyspark
findspark.find()

from pyspark.sql import SparkSession
from pyspark.sql.functions import count

spark = (SparkSession
         .builder
         .appName("Classification")
         .getOrCreate())

### Đọc và load tập dữ liệu Iris

In [2]:
irisDF = (spark.read
          .option("HEADER", True)
          .option("inferSchema", True)
          .csv("./data/iris.csv")
         )

irisDF.show(5)

+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|      class|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows


### Chuyển cột `class` (kiểu string) thành `label` (kiểu double)

In [3]:
from pyspark.ml.feature import StringIndexer

class_indexer = StringIndexer(inputCol = 'class', outputCol = 'label')

irisDFindexed = class_indexer.fit(irisDF).transform(irisDF)

irisDFindexed.show(5)

+------------+-----------+------------+-----------+-----------+-----+
|sepal_length|sepal_width|petal_length|petal_width|      class|label|
+------------+-----------+------------+-----------+-----------+-----+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|  0.0|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|  0.0|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|  0.0|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|  0.0|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|  0.0|
+------------+-----------+------------+-----------+-----------+-----+
only showing top 5 rows


### Tập dữ liệu Iris

`sepal_length`: chiều dài đài hoa (cm)

`sepal_width`: chiều rộng đài hoa (cm)

`petal_length`: chiều dài cánh hoa (cm)

`petal_width`: chiều rộng cánh hoa (cm)

`class/label`: loại hoa

![Iris dataset](./image/iris.png)

### Chia dữ liệu thành train/test set

In [4]:
(trainDF, testDF) = irisDFindexed.randomSplit([.8, .2], seed = 1)

### Xem các loại biến trong tập dữ liệu

In [5]:
irisDFindexed.dtypes

[('sepal_length', 'double'),
 ('sepal_width', 'double'),
 ('petal_length', 'double'),
 ('petal_width', 'double'),
 ('class', 'string'),
 ('label', 'double')]

### Biến đổi train data và test data theo định dạng của Spark

In [6]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols = ['sepal_length','sepal_width','petal_length','petal_width'],
                            outputCol = 'features')
assembler_train = assembler.transform(trainDF)

X_train = assembler_train.select('features', 'label')
X_train.show(5)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[4.3,3.0,1.1,0.1]|  0.0|
|[4.4,2.9,1.4,0.2]|  0.0|
|[4.4,3.0,1.3,0.2]|  0.0|
|[4.4,3.2,1.3,0.2]|  0.0|
|[4.6,3.1,1.5,0.2]|  0.0|
+-----------------+-----+
only showing top 5 rows


## Sử dụng Logistic Regression

### 1.1 Tạo mô hình Logistic Regression

Tạo một một hình Logistic Regression và huấn luyện mô hình trên `X_train` với `labelCol` là `'label'` và `featuresCol` là `'features'`

In [7]:
from pyspark.ml.classification import LogisticRegression

logit = LogisticRegression(featuresCol = "features", labelCol = "label")

logitModel = logit.fit(X_train)

### 1.2. Áp dụng mô hình trên test data

Áp dụng biến đổi cho tập test tương tự như trên tập train. In ra vài dòng sau khi biến đổi để xem kết quả.

In [8]:
assembler_test = assembler.transform(testDF)
X_test = assembler_test.select('features', 'label')
X_test.show(5)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[4.5,2.3,1.3,0.3]|  0.0|
|[4.8,3.1,1.6,0.2]|  0.0|
|[4.8,3.4,1.6,0.2]|  0.0|
|[4.8,3.4,1.9,0.2]|  0.0|
|[4.9,2.5,4.5,1.7]|  2.0|
+-----------------+-----+
only showing top 5 rows


Dự đoán trên test data

In [9]:
predictions = logitModel.transform(X_test)
predictions.select("prediction", "label").show(5)

+----------+-----+
|prediction|label|
+----------+-----+
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       1.0|  2.0|
+----------+-----+
only showing top 5 rows


### 1.3. Đánh giá mô hình

Tính giá trị `Accuracy` của mô hình trên tập test

In [10]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator()

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
print("Accuracy = %g" % accuracy)
print("Test Error = %g" % (1.0 - accuracy))

Accuracy = 0.961538
Test Error = 0.0384615


### 1.4. Tạo ML pipeline và đánh giá dùng phương pháp cross validation

![cross-validation-model-selection](./image/cross_val.png)

In [11]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import pprint

pp = pprint.PrettyPrinter(indent = 4)

# Create a LogisticRegression instance. This instance is an Estimator.
logit = LogisticRegression(featuresCol = "features", labelCol = "label")

# Define indexer
indexer = StringIndexer(inputCol = 'class', 
                        outputCol = 'label')

# Define assembler
assembler = VectorAssembler(inputCols = ['sepal_length','sepal_width','petal_length','petal_width'],
                            outputCol = 'features')

# Configure an ML pipeline, which consists of two stages: indexer, assembler, and logit.
pipeline = Pipeline(stages = [indexer, assembler, logit])

# Specify evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol = "label", 
    predictionCol = "prediction",
    metricName = "accuracy"
)

# Specify parameters
paramGrid = (ParamGridBuilder()
            .addGrid(logit.regParam , [0.01, 0.1, 1])
            .build())

# Train/test split
(trainDF, testDF) = irisDF.randomSplit([.8, .2], seed = 1)

# Setup CrossValidator 
# A CrossValidator requires an Estimator, a set of Estimator ParamMaps, and an Evaluator.
cv = CrossValidator(estimator = logit, 
                    evaluator = evaluator, 
                    estimatorParamMaps = paramGrid, 
                    numFolds = 3, 
                    parallelism = 2, 
                    seed = 1)

# Run cross-validation on training data, and choose the best set of parameters
logitModel = pipeline.fit(trainDF)

# Make predictions on test data. logitModel uses the best model found (regParam = 0.1)
prediction = logitModel.transform(testDF)
result = prediction.select("features", "label", "prediction").collect()

# Print some predictions
for row in result[0:5]:
    pp.pprint("features=%s, label=%s -> prediction=%s" % 
              (row.features, row.label, row.prediction))

accuracy = evaluator.evaluate(predictions)

print("Test Error = %g" % (1.0 - accuracy))

'features=[4.5,2.3,1.3,0.3], label=2.0 -> prediction=2.0'
'features=[4.8,3.1,1.6,0.2], label=2.0 -> prediction=2.0'
'features=[4.8,3.4,1.6,0.2], label=2.0 -> prediction=2.0'
'features=[4.8,3.4,1.9,0.2], label=2.0 -> prediction=2.0'
'features=[4.9,2.5,4.5,1.7], label=1.0 -> prediction=0.0'
Test Error = 0.0384615


# II. Áp dụng

## Câu 1 - Áp dụng `LogisticRegression` với tập dữ liệu `Auto`

Câu hỏi này sử dụng Logistic Regression trên tập dữ liệu `Auto` để dự đoán một xe cho trước có `mpg` là `high` hay `low`.

**Auto Data Set Description**

A data frame with 392 observations on the following 9 variables.

- `mpg`: miles per gallon

- `cylinders`: Number of cylinders between 4 and 8

- `displacement`: Engine displacement (cu. inches)

- `horsepower`: Engine horsepower

- `weight`: Vehicle weight (lbs.)

- `acceleration`: Time to accelerate from 0 to 60 mph (sec.)

- `year`: Model year (modulo 100)

- `origin`: Origin of car (1. American, 2. European, 3. Japanese)

- `name`: Vehicle name

**1.1.** Tạo một binary variable nhận giá trị 1 (`high`) với các xe có `mpg` lớn hơn median mpg, và nhận giá trị 0 (`low`) cho các xe còn lại.

In [13]:
# Viết code của bạn ở đây
from pyspark.sql import functions as F

# 1) Đọc dữ liệu Auto
auto_raw = spark.read.option("header", True).csv("./data/Auto.csv")

# 2) Loại bỏ dòng có horsepower = "?" và ép kiểu các cột cần dùng
auto = (
    auto_raw.filter(F.col("horsepower") != "?")
            .withColumn("mpg", F.col("mpg").cast("double"))
            .withColumn("cylinders", F.col("cylinders").cast("int"))
            .withColumn("displacement", F.col("displacement").cast("double"))
            .withColumn("horsepower", F.col("horsepower").cast("double"))
            .withColumn("weight", F.col("weight").cast("double"))
            .withColumn("acceleration", F.col("acceleration").cast("double"))
            .withColumn("year", F.col("year").cast("int"))
            .withColumn("origin", F.col("origin").cast("int"))
)

# 3) Tính median mpg bằng percentile_approx
median_mpg = auto.select(F.expr("percentile_approx(mpg, 0.5)")).first()[0]
print("Median mpg =", median_mpg)

# 4) Tạo biến nhãn nhị phân: mpg > median -> 1.0, ngược lại -> 0.0
auto = auto.withColumn("label", (F.col("mpg") > F.lit(median_mpg)).cast("double"))
auto.select("mpg", "label").show(10)

# 5) Chia train/test
train_auto, test_auto = auto.randomSplit([0.7, 0.3], seed=42)
print("Train =", train_auto.count(), "Test =", test_auto.count())


Median mpg = 22.5
+----+-----+
| mpg|label|
+----+-----+
|18.0|  0.0|
|15.0|  0.0|
|18.0|  0.0|
|16.0|  0.0|
|17.0|  0.0|
|15.0|  0.0|
|14.0|  0.0|
|14.0|  0.0|
|14.0|  0.0|
|15.0|  0.0|
+----+-----+
only showing top 10 rows
Train = 296 Test = 96


**1.2.** Áp dụng Logistic Regression cho tập dữ liệu với các giá trị siêu tham số `regParam` khác nhau để dự đoán `mpg`. Cho biết cross-validation error ứng với các giá trị khác nhau của siêu tham số này. Nhận xét kết quả thu được. Tham khảo document về Logistic Regression của Spark ở [LogisticRegression](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html#pyspark.ml.classification.LogisticRegression).

In [15]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# --- 1. Cấu hình Feature Engineering ---
# Xác định các nhóm cột
cat_col = "origin"
num_cols = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "year"]

# Tạo các bước xử lý dữ liệu (Stages)
# Chuyển đổi chuỗi sang chỉ số (StringIndexer)
stage_indexer = StringIndexer(
    inputCol=cat_col, outputCol="origin_idx", handleInvalid="keep"
)

# Mã hóa One-Hot (OneHotEncoder)
stage_encoder = OneHotEncoder(
    inputCols=["origin_idx"], outputCols=["origin_vec"], handleInvalid="keep"
)

# Gom tất cả đặc trưng vào một vector (VectorAssembler)
input_features = num_cols + ["origin_vec"]
stage_assembler = VectorAssembler(
    inputCols=input_features, outputCol="features", handleInvalid="keep"
)

# Khởi tạo mô hình phân loại
log_reg = LogisticRegression(featuresCol="features", labelCol="label", maxIter=200)

# Đóng gói vào Pipeline
ml_pipeline = Pipeline(stages=[stage_indexer, stage_encoder, stage_assembler, log_reg])

# --- 2. Thiết lập Grid Search & Cross Validation ---
# Định nghĩa thước đo đánh giá (Accuracy)
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

# Xây dựng lưới tham số cho regParam
reg_values = [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
param_grid = ParamGridBuilder().addGrid(log_reg.regParam, reg_values).build()

# Cấu hình CrossValidator
cv_runner = CrossValidator(
    estimator=ml_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=accuracy_eval,
    numFolds=5,
    parallelism=2,
    seed=42,
)

# --- 3. Huấn luyện mô hình ---
print("Đang huấn luyện Cross-Validation...")
cv_model = cv_runner.fit(train_auto)

# --- 4. Tổng hợp và hiển thị kết quả ---
# Trích xuất kết quả accuracy trung bình từ các folds
mean_scores = cv_model.avgMetrics

# Tạo danh sách kết quả để hiển thị
metrics_data = []
for reg_val, score in zip(reg_values, mean_scores):
    error_rate = 1.0 - score
    metrics_data.append((float(reg_val), float(score), float(error_rate)))

# Tạo DataFrame báo cáo
report_df = spark.createDataFrame(metrics_data, ["RegParam", "Accuracy", "Error_Rate"])
report_df.orderBy("RegParam").show(truncate=False)

# --- 5. Kiểm tra trên tập Test ---
# Lấy ra mô hình tốt nhất
optimal_model = cv_model.bestModel

# Dự đoán trên tập test
test_predictions = optimal_model.transform(test_auto)

# Tính toán độ chính xác cuối cùng
final_acc = accuracy_eval.evaluate(test_predictions)
final_err = 1.0 - final_acc

print(f"Kết quả trên tập Test:")
print(f"- Accuracy: {final_acc}")
print(f"- Error: {final_err}")

Đang huấn luyện Cross-Validation...
+--------+------------------+-------------------+
|RegParam|Accuracy          |Error_Rate         |
+--------+------------------+-------------------+
|0.0     |0.8987098690893758|0.10129013091062422|
|0.001   |0.8924537596074599|0.10754624039254013|
|0.01    |0.8986034922277806|0.10139650777221942|
|0.05    |0.9028524332509154|0.09714756674908465|
|0.1     |0.8999112567803271|0.10008874321967287|
|0.5     |0.896880953750024 |0.10311904624997603|
|1.0     |0.896880953750024 |0.10311904624997603|
+--------+------------------+-------------------+

Kết quả trên tập Test:
- Accuracy: 0.9270833333333334
- Error: 0.07291666666666663


## Câu 2 - So sánh các mô hình phân loại

- Thực hiện việc train tất cả các mô hình `LogisticRegression`, `DecisionTreeClassifier` và `RandomForestClassifier`, `GBTClassifier`, `MultilayerPerceptronClassifier`, `LinearSVC`, `NaiveBayes` trên tập dữ liệu HeartDisease (https://archive.ics.uci.edu/ml/datasets/heart+Disease) dùng độ đo Accuracy.

- Điều chỉnh các siêu tham số của các mô hình để chọn mô hình tốt nhất dùng cross validation (tham khảo mục 1.4 ở trên). Để tránh lặp lại các bước xử lý giống nhau nhiều lần như ở trên bạn cần tạo pipeline các bước xử lý. Tham khảo cách tạo pipeline cho mô hình ở https://spark.apache.org/docs/latest/ml-pipeline.html.

- So sánh và nhận xét về kết quả của các mô hình.

- Tham khảo document về các classifier của Spark ở 

    - Classification and Regression ở MLlib Guide: https://spark.apache.org/docs/latest/ml-classification-regression.html

    - Classification module: https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html#classification.

Bên dưới là một số gợi ý về khám phá sơ bộ và tiền xử lý dữ liệu.

In [16]:
heart = (spark.read
          .option("HEADER", True)
          .option("inferSchema", True)
          .csv("./data/HeartDisease.csv")
         )

heart.show(5)

+---+---+---+------------+------+----+---+-------+-----+-----+-------+-----+---+----------+---+
|_c0|Age|Sex|   ChestPain|RestBP|Chol|Fbs|RestECG|MaxHR|ExAng|Oldpeak|Slope| Ca|      Thal|AHD|
+---+---+---+------------+------+----+---+-------+-----+-----+-------+-----+---+----------+---+
|  1| 63|  1|     typical|   145| 233|  1|      2|  150|    0|    2.3|    3|  0|     fixed| No|
|  2| 67|  1|asymptomatic|   160| 286|  0|      2|  108|    1|    1.5|    2|  3|    normal|Yes|
|  3| 67|  1|asymptomatic|   120| 229|  0|      2|  129|    1|    2.6|    2|  2|reversable|Yes|
|  4| 37|  1|  nonanginal|   130| 250|  0|      0|  187|    0|    3.5|    3|  0|    normal| No|
|  5| 41|  0|  nontypical|   130| 204|  0|      2|  172|    0|    1.4|    1|  0|    normal| No|
+---+---+---+------------+------+----+---+-------+-----+-----+-------+-----+---+----------+---+
only showing top 5 rows


In [17]:
heart.count()

303

In [18]:
len(heart.columns)

15

In [19]:
heart.dtypes

[('_c0', 'int'),
 ('Age', 'int'),
 ('Sex', 'int'),
 ('ChestPain', 'string'),
 ('RestBP', 'int'),
 ('Chol', 'int'),
 ('Fbs', 'int'),
 ('RestECG', 'int'),
 ('MaxHR', 'int'),
 ('ExAng', 'int'),
 ('Oldpeak', 'double'),
 ('Slope', 'int'),
 ('Ca', 'string'),
 ('Thal', 'string'),
 ('AHD', 'string')]

In [20]:
heart.withColumn('Ca', heart.Ca.cast('int'))

DataFrame[_c0: int, Age: int, Sex: int, ChestPain: string, RestBP: int, Chol: int, Fbs: int, RestECG: int, MaxHR: int, ExAng: int, Oldpeak: double, Slope: int, Ca: int, Thal: string, AHD: string]

In [21]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import OneHotEncoder

# using one-hot-encoding (OHE) for categorical variables
catInputCols = ['ChestPain', 'RestECG', 'Slope', 'Thal']
catOutputCols = [x + "Index" for x in catInputCols]
oheCatOutputCols = [x + "OHE" for x in catInputCols]

catIndexer = StringIndexer(inputCols = catInputCols, outputCols = catOutputCols)

catOHEncoder = OneHotEncoder(inputCols = catOutputCols, 
                          outputCols = oheCatOutputCols)

pipeline = Pipeline(stages = [catIndexer, catOHEncoder])

# fit the pipeline model and transform the data as defined
pipelineModel = pipeline.fit(heart)

# view the transformed data
transformed_heart = pipelineModel.transform(heart)
transformed_heart.select(catInputCols + oheCatOutputCols).show(5)

+------------+-------+-----+----------+-------------+-------------+-------------+-------------+
|   ChestPain|RestECG|Slope|      Thal| ChestPainOHE|   RestECGOHE|     SlopeOHE|      ThalOHE|
+------------+-------+-----+----------+-------------+-------------+-------------+-------------+
|     typical|      2|    3|     fixed|    (3,[],[])|(2,[1],[1.0])|    (2,[],[])|(3,[2],[1.0])|
|asymptomatic|      2|    2|    normal|(3,[0],[1.0])|(2,[1],[1.0])|(2,[1],[1.0])|(3,[0],[1.0])|
|asymptomatic|      2|    2|reversable|(3,[0],[1.0])|(2,[1],[1.0])|(2,[1],[1.0])|(3,[1],[1.0])|
|  nonanginal|      0|    3|    normal|(3,[1],[1.0])|(2,[0],[1.0])|    (2,[],[])|(3,[0],[1.0])|
|  nontypical|      2|    1|    normal|(3,[2],[1.0])|(2,[1],[1.0])|(2,[0],[1.0])|(3,[0],[1.0])|
+------------+-------+-----+----------+-------------+-------------+-------------+-------------+
only showing top 5 rows


In [27]:
# Viết code của bạn ở đây (bạn có thể tạo thêm các cell khác để thực nghiệm và phân tích kết quả)
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
    MultilayerPerceptronClassifier,
    LinearSVC,
    NaiveBayes,
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col


# 1. Định nghĩa các cột
cat_cols = ["Sex", "ChestPain", "Fbs", "RestECG", "ExAng", "Slope", "Ca", "Thal"]
num_cols = ["Age", "RestBP", "Chol", "MaxHR", "Oldpeak"]
label_col = "AHD"  # cột nhãn thực tế trong bộ HeartDisease

# 2. Xây dựng các stages cho Feature Engineering Pipeline
stages = []

# Xử lý biến phân loại: StringIndexer -> OneHotEncoder
for c in cat_cols:
    indexer = StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[c + "_idx"], outputCols=[c + "_vec"])
    stages += [indexer, encoder]

# Gom tất cả features vào một vector
assembler_inputs = [c + "_vec" for c in cat_cols] + num_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features")
stages.append(assembler)

# Chuẩn hóa dữ liệu (StandardScaler) - Quan trọng cho LogisticRegression, LinearSVC, MLP
# Lưu ý: NaiveBayes có thể yêu cầu dữ liệu không âm, nên ta có thể cần xử lý riêng hoặc dùng MinMaxScaler.
# Tuy nhiên, để đơn giản cho pipeline chung, ta dùng StandardScaler (có thể gây lỗi cho NaiveBayes Multinomial mặc định nếu có giá trị âm).
# Ở đây ta dùng StandardScaler với withMean=False để giữ tính thưa (sparsity) và tránh giá trị âm nếu dữ liệu gốc dương.
scaler = StandardScaler(
    inputCol="raw_features", outputCol="features", withStd=True, withMean=False
)
stages.append(scaler)

# Index label nếu cần (đảm bảo label là 0, 1...)
label_indexer = StringIndexer(
    inputCol=label_col, outputCol="label"
)
stages.append(label_indexer)

# Tạo Pipeline xử lý dữ liệu cơ sở
prep_pipeline = Pipeline(stages=stages)

# Fit và Transform dữ liệu để chuẩn bị cho việc xác định input layer của MLP
# Chia tập dữ liệu train/test (80/20)
train_data, test_data = heart.randomSplit([0.8, 0.2], seed=42)

# Fit pipeline tiền xử lý trên tập train
prep_model = prep_pipeline.fit(train_data)
train_prep = prep_model.transform(train_data)
test_prep = prep_model.transform(test_data)

# Xác định kích thước input feature cho MultilayerPerceptronClassifier
input_dim = len(train_prep.select("features").first()[0])
print(f"Input Feature Dimension: {input_dim}")

Input Feature Dimension: 30


In [28]:
# Định nghĩa Evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

# Cấu hình các mô hình và ParamGrid
models_config = []

# 1. Logistic Regression
lr = LogisticRegression(labelCol="label", featuresCol="features")
lr_param = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
models_config.append(("LogisticRegression", lr, lr_param))

# 2. Decision Tree
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")
dt_param = (
    ParamGridBuilder().addGrid(dt.maxDepth, [5, 10]).addGrid(dt.maxBins, [32]).build()
)
models_config.append(("DecisionTreeClassifier", dt, dt_param))

# 3. Random Forest
rf = RandomForestClassifier(labelCol="label", featuresCol="features")
rf_param = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 20])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)
models_config.append(("RandomForestClassifier", rf, rf_param))

# 4. GBT Classifier (Gradient Boosted Trees)
gbt = GBTClassifier(labelCol="label", featuresCol="features")
gbt_param = (
    ParamGridBuilder().addGrid(gbt.maxIter, [10, 20]).addGrid(gbt.maxDepth, [5]).build()
)
models_config.append(("GBTClassifier", gbt, gbt_param))

# 5. Multilayer Perceptron Classifier
# Layers: [input, hidden1, hidden2, output]
# Output size = 2 (binary classification)
layers = [input_dim, 16, 8, 2]
mlp = MultilayerPerceptronClassifier(
    labelCol="label", featuresCol="features", layers=layers, blockSize=128, seed=1234
)
mlp_param = ParamGridBuilder().addGrid(mlp.maxIter, [100]).build()
models_config.append(("MultilayerPerceptronClassifier", mlp, mlp_param))

# 6. Linear SVC
lsvc = LinearSVC(labelCol="label", featuresCol="features")
lsvc_param = (
    ParamGridBuilder()
    .addGrid(lsvc.maxIter, [10, 50])
    .addGrid(lsvc.regParam, [0.01, 0.1])
    .build()
)
models_config.append(("LinearSVC", lsvc, lsvc_param))

# 7. Naive Bayes
# Lưu ý: NaiveBayes mặc định là "multinomial" yêu cầu features không âm.
# Nếu StandardScaler(withMean=False) giữ nguyên dấu dương thì ổn.
# Nếu không, nên dùng modelType="gaussian" cho dữ liệu số thực đã chuẩn hóa.
nb = NaiveBayes(labelCol="label", featuresCol="features", modelType="gaussian")
nb_param = ParamGridBuilder().addGrid(nb.smoothing, [0.0, 1.0]).build()
models_config.append(("NaiveBayes", nb, nb_param))

In [29]:
results = []

print(f"{'Model':<30} | {'Best Accuracy':<15} | {'Status'}")
print("-" * 60)

for name, model, param_grid in models_config:
    try:
        # Tạo CrossValidator
        cv = CrossValidator(
            estimator=model,
            estimatorParamMaps=param_grid,
            evaluator=evaluator,
            numFolds=3,  # Số fold kiểm thử chéo
            seed=42,
        )

        # Huấn luyện trên tập dữ liệu đã tiền xử lý (train_prep)
        cv_model = cv.fit(train_prep)

        # Đánh giá trên tập test (test_prep)
        best_model = cv_model.bestModel
        predictions = best_model.transform(test_prep)
        accuracy = evaluator.evaluate(predictions)

        results.append((name, accuracy))
        print(f"{name:<30} | {accuracy:.4f}          | Done")

    except Exception as e:
        print(f"{name:<30} | Error            | {str(e)[:50]}...")

# Hiển thị bảng kết quả tổng hợp
print("\n--- TỔNG HỢP KẾT QUẢ ---")
result_df = spark.createDataFrame(results, ["Model", "Accuracy"]).orderBy(
    col("Accuracy").desc()
)
result_df.show(truncate=False)

Model                          | Best Accuracy   | Status
------------------------------------------------------------
LogisticRegression             | 0.7872          | Done
DecisionTreeClassifier         | 0.6383          | Done
RandomForestClassifier         | 0.7660          | Done
GBTClassifier                  | 0.6596          | Done
MultilayerPerceptronClassifier | 0.6596          | Done
LinearSVC                      | 0.7660          | Done
NaiveBayes                     | 0.6383          | Done

--- TỔNG HỢP KẾT QUẢ ---
+------------------------------+------------------+
|Model                         |Accuracy          |
+------------------------------+------------------+
|LogisticRegression            |0.7872340425531915|
|RandomForestClassifier        |0.7659574468085106|
|LinearSVC                     |0.7659574468085106|
|GBTClassifier                 |0.6595744680851063|
|MultilayerPerceptronClassifier|0.6595744680851063|
|DecisionTreeClassifier        |0.6382978723404

### Nhận xét và so sánh kết quả:

- **Mô hình tốt nhất:** **Logistic Regression** đạt độ chính xác cao nhất (**~78.7%**), cho thấy đây là mô hình phù hợp nhất với bộ dữ liệu này trong các tham số đã thử nghiệm.

- **Nhóm hiệu suất cao:** **Random Forest** và **LinearSVC** cùng đạt độ chính xác **~76.6%**.

- **Nhóm hiệu suất thấp hơn:** **GBTClassifier** và **MultilayerPerceptronClassifier** chỉ đạt khoảng **66%**. Điều này có thể do các mô hình này phức tạp, cần nhiều dữ liệu hơn để huấn luyện hoặc cần tinh chỉnh siêu tham số kỹ hơn (learning rate, số layers, v.v.) để tránh overfitting/underfitting.

- **Mô hình cơ sở:** **Decision Tree** và **Naive Bayes** có kết quả thấp nhất (**~63.8%**). Decision Tree đơn lẻ thường yếu hơn so với bản phối hợp (Random Forest), còn Naive Bayes có thể bị ảnh hưởng bởi sự tương quan giữa các đặc trưng (ví dụ: các chỉ số sức khỏe thường liên quan đến nhau).